# Worker validation and rasterization with LocalStack

This notebook is for contributors testing the validation and rasterization worker slices against the repository's local AWS boundary. It uses LocalStack S3 for source and page artifacts and LocalStack DynamoDB for persisted failure states.

Start the pinned service before running the notebook:

```bash
docker compose up -d localstack
```

Run the notebook from the repository's Python 3.12 environment. It creates reusable `redact-notebook-*` resources and clears only its own job records and S3 prefixes on each run.


## Outline

1. Connect to LocalStack and provision notebook resources
2. Generate and upload PDF, JPEG, and PNG fixtures
3. Read sources from S3 and validate them
4. Rasterize into S3 and inspect the artifact contract
5. Confirm sequential page writes
6. Persist validation and processing failures in DynamoDB
7. Try a DPI experiment


In [ ]:
from __future__ import annotations

import io
import json
import sys
from collections.abc import Iterable
from datetime import UTC, datetime, timedelta
from pathlib import Path

import boto3
import pypdfium2 as pdfium
from botocore.config import Config
from botocore.exceptions import EndpointConnectionError
from PIL import Image

repo_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src" / "redact_api").is_dir()
)
sys.path.insert(0, str(repo_root / "src"))

from redact_api.aws import DynamoJobRepository, S3ObjectRepository
from redact_api.domain import FailureCode, Job, JobStatus, SourceType
from redact_api.worker import (
    RenderedPage,
    ValidatedSource,
    WorkerRasterizationConfig,
    analyze_source,
    rasterize_source,
    validate_source,
)

NOW = datetime(2026, 6, 18, 12, 0, tzinfo=UTC)
repo_root


## 1. Connect to LocalStack

These settings match `tests/conftest.py`. The S3 client uses path-style addressing, which keeps generated URLs and requests compatible with the local endpoint.


In [ ]:
LOCALSTACK_URL = "http://localhost:4566"
REGION = "eu-west-1"
TABLE_NAME = "redact-notebook-jobs"
BUCKET_NAME = "redact-notebook-files"

aws = {
    "endpoint_url": LOCALSTACK_URL,
    "region_name": REGION,
    "aws_access_key_id": "test",
    "aws_secret_access_key": "test",
}
dynamodb = boto3.resource("dynamodb", **aws)
s3 = boto3.client(
    "s3",
    config=Config(s3={"addressing_style": "path"}),
    **aws,
)

try:
    localstack_health = s3.list_buckets()["ResponseMetadata"]["HTTPStatusCode"]
except EndpointConnectionError as error:
    raise RuntimeError(
        "LocalStack is unavailable; run `docker compose up -d localstack`."
    ) from error

assert localstack_health == 200
{"endpoint": LOCALSTACK_URL, "status": localstack_health}


In [ ]:
existing_tables = dynamodb.meta.client.list_tables()["TableNames"]
if TABLE_NAME not in existing_tables:
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "id", "KeyType": "HASH"}],
        AttributeDefinitions=[
            {"AttributeName": "id", "AttributeType": "S"},
            {"AttributeName": "owner_id", "AttributeType": "S"},
            {"AttributeName": "created_sort", "AttributeType": "S"},
        ],
        GlobalSecondaryIndexes=[
            {
                "IndexName": "owner-created-index",
                "KeySchema": [
                    {"AttributeName": "owner_id", "KeyType": "HASH"},
                    {"AttributeName": "created_sort", "KeyType": "RANGE"},
                ],
                "Projection": {"ProjectionType": "ALL"},
            }
        ],
        BillingMode="PAY_PER_REQUEST",
    )
    table.wait_until_exists()
else:
    table = dynamodb.Table(TABLE_NAME)

existing_buckets = {item["Name"] for item in s3.list_buckets()["Buckets"]}
if BUCKET_NAME not in existing_buckets:
    s3.create_bucket(
        Bucket=BUCKET_NAME,
        CreateBucketConfiguration={"LocationConstraint": REGION},
    )

jobs = DynamoJobRepository(table)
objects = S3ObjectRepository(s3, bucket=BUCKET_NAME)
{"table": table.table_status, "bucket": BUCKET_NAME}


The worker slice exposes a small `PageArtifactWriter` protocol but does not yet ship an S3 implementation. This notebook adapter maps that protocol directly to LocalStack S3 and records write order for inspection.


In [ ]:
class S3PageArtifactWriter:
    def __init__(self, client, bucket: str) -> None:
        self.client = client
        self.bucket = bucket
        self.events: list[tuple[str, str]] = []

    def put_page_artifact(self, key: str, content: bytes) -> None:
        self.client.put_object(
            Bucket=self.bucket, Key=key, Body=content, ContentType="image/png"
        )
        self.events.append(("page", key))

    def put_page_artifact_index(self, key: str, content: bytes) -> None:
        self.client.put_object(
            Bucket=self.bucket,
            Key=key,
            Body=content,
            ContentType="application/json",
        )
        self.events.append(("index", key))


def read_s3(key: str, *, bucket: str = BUCKET_NAME) -> bytes:
    return s3.get_object(Bucket=bucket, Key=key)["Body"].read()


def clear_job(job_id: str) -> None:
    table.delete_item(Key={"id": job_id})
    prefix = f"users/notebook-user/jobs/{job_id}/"
    contents = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix).get(
        "Contents", []
    )
    if contents:
        s3.delete_objects(
            Bucket=BUCKET_NAME,
            Delete={"Objects": [{"Key": item["Key"]} for item in contents]},
        )


## 2. Generate and upload source fixtures

The fixtures are deterministic and contain no sensitive data. Each source gets its own owner/job namespace so all artifacts can coexist in S3.


In [ ]:
def make_job(job_id: str, source_type: SourceType) -> Job:
    prefix = f"users/notebook-user/jobs/{job_id}/"
    return Job(
        id=job_id,
        owner_id="notebook-user",
        source_type=source_type,
        source_key=f"{prefix}source",
        output_key=f"{prefix}output",
        status=JobStatus.QUEUED,
        page_count=0,
        completed_pages=0,
        created_at=NOW,
        expires_at=NOW + timedelta(hours=24),
        model_versions={},
        failure_code=None,
        version=1,
    )


def make_pdf(page_sizes: tuple[tuple[int, int], ...]) -> bytes:
    document = pdfium.PdfDocument.new()
    for width, height in page_sizes:
        document.new_page(width, height)
    output = io.BytesIO()
    document.save(output)
    document.close()
    return output.getvalue()


def make_image(source_type: SourceType, size: tuple[int, int]) -> bytes:
    output = io.BytesIO()
    image = Image.new("RGB", size, color=(210, 45, 70))
    image.save(output, format="JPEG" if source_type is SourceType.JPEG else "PNG")
    return output.getvalue()


fixtures = {
    "notebook-pdf": (SourceType.PDF, make_pdf(((72, 72), (144, 72)))),
    "notebook-jpeg": (SourceType.JPEG, make_image(SourceType.JPEG, (24, 16))),
    "notebook-png": (SourceType.PNG, make_image(SourceType.PNG, (20, 12))),
}
source_jobs = {}
for job_id, (source_type, content) in fixtures.items():
    clear_job(job_id)
    job = make_job(job_id, source_type)
    s3.put_object(Bucket=BUCKET_NAME, Key=job.source_key, Body=content)
    assert objects.object_exists(job.source_key)
    source_jobs[job_id] = job

{job_id: job.source_key for job_id, job in source_jobs.items()}


In [ ]:
download_dir = repo_root / "tmp" / "notebooks" / "downloads"
download_dir.mkdir(parents=True, exist_ok=True)

for obj in s3.list_objects_v2(Bucket="redact-notebook-files", Prefix="")["Contents"]:
    print(obj["Key"])
    download_path = download_dir / Path(obj["Key"]).name
    s3.download_file("redact-notebook-files", obj["Key"], str(download_path))


## 3. Read from S3 and validate

Validation receives the exact bytes returned by LocalStack S3. PDF page count is two; JPEG and PNG inputs are one-page sources.


In [ ]:
source_bytes = {
    job_id: read_s3(job.source_key) for job_id, job in source_jobs.items()
}
validated = {
    job_id: validate_source(job.source_type, source_bytes[job_id])
    for job_id, job in source_jobs.items()
}
[
    {"job_id": job_id, "source_type": item.source_type.value, "pages": item.page_count}
    for job_id, item in validated.items()
]


## 4. Rasterize into S3

At 72 DPI, PDF points map directly to pixels. Pages are written first, followed by one deterministic JSON index under the same owner/job namespace.


In [ ]:
pdf_job = source_jobs["notebook-pdf"]
pdf_writer = S3PageArtifactWriter(s3, BUCKET_NAME)
pdf_index = rasterize_source(
    pdf_job,
    source_bytes[pdf_job.id],
    validated[pdf_job.id],
    writer=pdf_writer,
    config=WorkerRasterizationConfig(dpi=72),
)
stored_index = json.loads(read_s3(pdf_index.key))
{
    "events": pdf_writer.events,
    "pages": [
        {"number": page.page_number, "size": (page.width, page.height), "key": page.key}
        for page in pdf_index.pages
    ],
    "stored_index": stored_index,
}


In [ ]:
# Jupyter renders these LocalStack-backed PNG objects as previews.
[Image.open(io.BytesIO(read_s3(page.key))) for page in pdf_index.pages]


JPEG and PNG inputs normalize to the same one-page PNG contract. The checks below inspect object metadata and bytes directly from LocalStack.


In [ ]:
image_results = {}
for job_id in ("notebook-jpeg", "notebook-png"):
    job = source_jobs[job_id]
    writer = S3PageArtifactWriter(s3, BUCKET_NAME)
    index = rasterize_source(
        job, source_bytes[job_id], validated[job_id], writer=writer
    )
    page = index.pages[0]
    response = s3.get_object(Bucket=BUCKET_NAME, Key=page.key)
    content = response["Body"].read()
    image_results[job.source_type.value] = {
        "page_count": len(index.pages),
        "size": (page.width, page.height),
        "content_type": response["ContentType"],
        "is_png": content.startswith(b"\x89PNG\r\n\x1a\n"),
    }
image_results


## 5. Confirm sequential page writes

This isolated probe intentionally uses an in-memory recorder: it must observe the exact moment the generator requests page 2, which an S3 listing cannot expose. Production rasterization above still writes its actual artifacts to LocalStack.


In [ ]:
class OrderingWriter:
    def __init__(self) -> None:
        self.events: list[tuple[str, str]] = []

    def put_page_artifact(self, key: str, content: bytes) -> None:
        self.events.append(("page", key))

    def put_page_artifact_index(self, key: str, content: bytes) -> None:
        self.events.append(("index", key))


ordering_writer = OrderingWriter()
render_events: list[str] = []


def render_pages() -> Iterable[RenderedPage]:
    render_events.append("render-1")
    yield RenderedPage(1, 1, 1, b"\x89PNG\r\n\x1a\npage-1")
    assert ordering_writer.events[0][0] == "page"
    render_events.append("render-2")
    yield RenderedPage(2, 1, 1, b"\x89PNG\r\n\x1a\npage-2")


rasterize_source(
    pdf_job,
    b"unused by injected renderer",
    ValidatedSource(SourceType.PDF, 2),
    writer=ordering_writer,
    render_pages=render_pages,
)
{"renders": render_events, "writes": ordering_writer.events}


## 6. Persist failures in DynamoDB

First, a source declared as PDF but containing PNG bytes is rejected before rasterization. `analyze_source` saves the permanent `unsupported_content` state through the real DynamoDB repository.


In [ ]:
validation_job = make_job("notebook-validation-failure", SourceType.PDF)
clear_job(validation_job.id)
jobs.create(validation_job)
validation_failed = analyze_source(
    validation_job,
    source_bytes["notebook-png"],
    jobs=jobs,
    now=NOW,
)
persisted_validation_failure = jobs.get(validation_job.id)
assert persisted_validation_failure == validation_failed
{
    "status": persisted_validation_failure.status.value,
    "failure_code": persisted_validation_failure.failure_code.value,
    "version": persisted_validation_failure.version,
}


Next, a missing LocalStack bucket causes an S3 write failure after successful PNG validation. The worker maps it to `processing_failed`, persists it in DynamoDB, and skips the downstream callback.


In [ ]:
processing_job = make_job("notebook-processing-failure", SourceType.PNG)
clear_job(processing_job.id)
jobs.create(processing_job)
s3.put_object(
    Bucket=BUCKET_NAME,
    Key=processing_job.source_key,
    Body=source_bytes["notebook-png"],
)
downstream_calls: list[ValidatedSource] = []
processing_failed = analyze_source(
    processing_job,
    read_s3(processing_job.source_key),
    jobs=jobs,
    now=NOW,
    raster_writer=S3PageArtifactWriter(s3, "missing-notebook-bucket"),
    write_artifacts=downstream_calls.append,
)
persisted_processing_failure = jobs.get(processing_job.id)
assert persisted_processing_failure == processing_failed
assert persisted_processing_failure.failure_code is FailureCode.PROCESSING_FAILED
assert not downstream_calls
{
    "status": persisted_processing_failure.status.value,
    "failure_code": persisted_processing_failure.failure_code.value,
    "version": persisted_processing_failure.version,
    "downstream_calls": len(downstream_calls),
}


## 7. Exercise: change rasterization DPI

Predict the dimensions of the first 72-by-72-point PDF page at 150 DPI, then run the cell. Change `exercise_dpi` and compare dimensions and S3 object sizes.

Common pitfall: pass source bytes and `ValidatedSource` metadata from the same S3 object; mixing them can produce a page-count mismatch. An optional extension is to inspect the notebook bucket with `aws --endpoint-url http://localhost:4566 s3 ls --recursive s3://redact-notebook-files`.


In [ ]:
exercise_dpi = 150  # Change this value and rerun.
exercise_writer = S3PageArtifactWriter(s3, BUCKET_NAME)
exercise_index = rasterize_source(
    pdf_job,
    source_bytes[pdf_job.id],
    validated[pdf_job.id],
    writer=exercise_writer,
    config=WorkerRasterizationConfig(dpi=exercise_dpi),
)
[
    {
        "page_number": page.page_number,
        "size": (page.width, page.height),
        "s3_bytes": s3.head_object(Bucket=BUCKET_NAME, Key=page.key)["ContentLength"],
    }
    for page in exercise_index.pages
]
